In [ ]:
# SPADIVA_REPRODUCIBILITY_SETUP
from pathlib import Path
import json
import os
import sys


def find_reproducibility_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "MANIFEST.tsv").exists() and (candidate / "spaDIVA").is_dir():
            return candidate
    raise RuntimeError("Could not find spaDIVA_reproducibility repository root.")


REPO_ROOT = find_reproducibility_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_config_path = REPO_ROOT / "config" / "paths.local.json"
_config = json.loads(_config_path.read_text(encoding="utf-8")) if _config_path.exists() else {}
DATA_ROOT = Path(os.environ.get("SPADIVA_DATA_ROOT", _config.get("data_root", "data"))).expanduser()
RESULTS_ROOT = Path(os.environ.get("SPADIVA_RESULTS_ROOT", _config.get("results_root", "results")))
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO_ROOT / DATA_ROOT
if not RESULTS_ROOT.is_absolute():
    RESULTS_ROOT = REPO_ROOT / RESULTS_ROOT
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("NUMBA_CACHE_DIR", str(REPO_ROOT / "_numba_cache"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / "_mpl_cache"))
Path(os.environ["NUMBA_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

for _rel in [
    "simulation", "simulation/spaDIVA",
    "P22", "P22/spaDIVA",
    "E13E15E18", "E13E15E18/spaDIVA",
    "P21P22", "P21P22/spaDIVA",
    "human lymph node", "human lymph node/spaDIVA",
]:
    (REPO_ROOT / _rel).mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")


# Lymph node benchmark and spatial maps: release spaDIVA, corr graph, KL1111, 200 epochs

This notebook runs the release `spaDIVA` package on the human lymph node RNA-ADT dataset using the current benchmark setting: correlation graph, `KL_weight=[1,1,1,1]`, and 200 training epochs. Outputs are written below `SPADIVA_RESULTS_ROOT`.


## 1. Imports and paths

This notebook uses the portability setup cell above to locate the repository root, data directory and output directory.

In [ ]:
import sys
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

RELEASE_ROOT = REPO_ROOT

from spaDIVA.analysis import fuse_shared_latent, make_integrated_embedding, collect_spadiva_outputs
from spaDIVA.train import train_spadiva, infer_latents
from spaDIVA.utils import cal_spatial, clr_normalize_each_cell, build_modality_graphs, cluster

warnings.filterwarnings("ignore")
sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

## 2. Benchmark settings

`K_VALUES=[10]` reproduces the full-embedding lymph benchmark configuration used here. Use `K_VALUES=[6, 10]` or `list(range(6, 11))` for the extended sensitivity analysis.

In [ ]:
MAX_EPOCHS = 200
K_VALUES = [10]
SEEDS = [202608010 + i for i in range(10)] 
# To reproduce the older 101000-101009 seed style instead, use:
# SEEDS = [101000 + i for i in range(10)]

KL_WEIGHT = [1, 1, 1, 1]
LEARNING_RATE = 1e-3
WEIGHT = 1.0
USE_CUDA = False

SPATIAL_K = 3
FEATURE_K = 20
SPATIAL_WEIGHT = 0.8
FEATURE_WEIGHT = 0.2
FEATURE_METRIC = "correlation"
WNN_K = 20

CLUSTER_METHOD = "mclust"
PLOT_EACH_RUN = False
RESUME_BENCHMARK_RESULTS = False


_dataset_roots = [
    DATA_ROOT / "datasets" / "human_lymph_node_RNA_ADT",
    DATA_ROOT / "human_lymph_node_RNA_ADT",
    DATA_ROOT,
]
DATA_DIR = next((root / "input_data" for root in _dataset_roots if (root / "input_data").is_dir()), None)
ANNOTATION_PATH = next((root / "annotation" / "annotation.csv" for root in _dataset_roots if (root / "annotation" / "annotation.csv").exists()), None)
if DATA_DIR is None or ANNOTATION_PATH is None:
    raise FileNotFoundError("Set SPADIVA_DATA_ROOT to the extracted Zenodo archive root.")
OUTPUT_DIR = RESULTS_ROOT / "human_lymph_node_RNA_ADT" / "benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = OUTPUT_DIR / "lymph_corr1111_epoch200_release_results.csv"
SUMMARY_PATH = OUTPUT_DIR / "lymph_corr1111_epoch200_release_summary.csv"
METADATA_PATH = OUTPUT_DIR / "lymph_corr1111_epoch200_release_metadata.json"
ARI_PDF = OUTPUT_DIR / "lymph_corr1111_epoch200_release_ari_boxplot.pdf"
NMI_PDF = OUTPUT_DIR / "lymph_corr1111_epoch200_release_nmi_boxplot.pdf"

PAPER_OUTPUT_DIR = RESULTS_ROOT / "human_lymph_node_RNA_ADT" / "spaDIVA"
PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAPER_BENCHMARK_CSV = PAPER_OUTPUT_DIR / "spaDIVA_benchmark_ari.csv"
PAPER_METRICS_TXT = PAPER_OUTPUT_DIR / "output_ari.txt"
SAVE_PUBLICATION_SPATIAL_MAPS = True
PAPER_SPATIAL_MAP_K = 10
PAPER_SPECIFIC_K = 5
PAPER_SPATIAL_MAP_SEED = "best_Z_W"

DATA_DIR


## 3. Helper functions

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def save_spatial_label_pdf(spatial, labels_by_name, out_path, s=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    n = len(labels_by_name)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (name, labels) in zip(axes, labels_by_name.items()):
        labels = pd.Series(labels).astype(str).values
        cats = sorted(pd.unique(labels), key=lambda x: int(float(x)) if str(x).replace('.', '', 1).isdigit() else str(x))
        cmap = plt.get_cmap("tab20")
        color_map = {cat: cmap(i % 20) for i, cat in enumerate(cats)}
        colors = [color_map[x] for x in labels]
        ax.scatter(spatial[:, 0], spatial[:, 1], c=colors, s=s, linewidths=0)
        ax.set_title(name)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def summarize_results(results_df):
    metric_cols = [c for c in results_df.columns if c.startswith("ARI_") or c.startswith("NMI_")]
    summary = (
        results_df.groupby("num_cluster")[metric_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    summary.columns = ["_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else str(col) for col in summary.columns]
    return summary


def save_metric_boxplot(results_df, metric_prefix, ylabel, out_pdf):
    keys = ["Z", "Z_W", "W_ADT", "W_RNA"]
    values = [results_df[f"{metric_prefix}_{key}"].dropna().to_numpy() for key in keys]
    fig, ax = plt.subplots(figsize=(2.8, 2.0))
    bp = ax.boxplot(
        values,
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=0.7),
        boxprops=dict(linewidth=0.6),
        whiskerprops=dict(linewidth=0.6),
        capprops=dict(linewidth=0.6),
    )
    colors = ["#4C78A8", "#E64B35", "#59A14F", "#F28E2B"]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    ax.set_xticks(range(1, len(keys) + 1))
    ax.set_xticklabels(keys, rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_pdf, bbox_inches="tight")
    fig.savefig(Path(out_pdf).with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_single_spatial_label_pdf(spatial, labels, out_path, s=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plot_adata = sc.AnnData(np.zeros((len(labels), 1)))
    plot_adata.obsm["spatial"] = spatial
    plot_adata.obs["mclust_diag"] = pd.Series(labels).astype(str).values
    sc.set_figure_params(figsize=(3, 3))
    ax = sc.pl.embedding(
        plot_adata,
        basis="spatial",
        color="mclust_diag",
        s=s,
        show=False,
        legend_loc="none",
    )
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    fig = ax.figure if hasattr(ax, "figure") else plt.gcf()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


## 4. Load and preprocess lymph node RNA-ADT data

This follows the release lymph tutorial preprocessing: RNA HVG + PCA and ADT CLR + PCA. PCA is deterministic to reduce preprocessing noise across repeated runs.

In [ ]:
PREPROCESS_SEED = 0
set_all_seeds(PREPROCESS_SEED)

adata_adt = sc.read(DATA_DIR / "human lymph node adata_ADT.h5ad")
adata_rna = sc.read(DATA_DIR / "human lymph node adata_RNA.h5ad")
annotation = pd.read_csv(ANNOTATION_PATH).set_index("Barcode")

adata_adt.var_names_make_unique()
adata_rna.var_names_make_unique()
adata_adt.X = adata_adt.X.astype("float32")
adata_rna.X = adata_rna.X.astype("float32")

adata_adt.obs["ground_truth"] = annotation.loc[adata_adt.obs_names, "manual-anno"]

sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)
sc.pp.scale(adata_rna)
adata_rna = adata_rna[:, adata_rna.var.highly_variable]

adata_adt = adata_adt[adata_rna.obs_names].copy()
adata_adt = clr_normalize_each_cell(adata_adt)
sc.pp.scale(adata_adt)

n_components = adata_adt.n_vars - 1
pca_adt = PCA(n_components=n_components, svd_solver="full")
adata_adt.obsm["clr"] = pca_adt.fit_transform(adata_adt.to_df())

pca_rna = PCA(n_components=n_components, svd_solver="randomized", random_state=PREPROCESS_SEED)
adata_rna.obsm["pca"] = pca_rna.fit_transform(adata_rna.to_df())

spatial = adata_adt.obsm["spatial"]
Y = adata_adt.obs["ground_truth"].values
X1_input = adata_adt.obsm["clr"]
X2_input = adata_rna.obsm["pca"]
X1_train = X1_input
X2_train = X2_input

adata_adt.shape, adata_rna.shape, X1_input.shape, X2_input.shape

## 5. Build corr graph

The graph uses spatial neighbors plus modality-specific feature neighbors with correlation distance.

In [ ]:
edge_index = cal_spatial(spatial, k=SPATIAL_K)
edge_index_fused1, edge_weight_fused1, edge_index_fused2, edge_weight_fused2 = build_modality_graphs(
    X1_input,
    X2_input,
    edge_index,
    k_feature=FEATURE_K,
    spatial_weight=SPATIAL_WEIGHT,
    feature_weight=FEATURE_WEIGHT,
    feature_metric=FEATURE_METRIC,
)

edge_index.shape, edge_index_fused1.shape, edge_index_fused2.shape

## 6. Run 10-seed benchmark

The cell saves a partial CSV after each seed, so you can interrupt and still keep completed runs.

In [ ]:
rows = []
current_targets = {(int(seed), int(k)) for seed in SEEDS for k in K_VALUES}
if RESUME_BENCHMARK_RESULTS and RAW_PATH.exists():
    old = pd.read_csv(RAW_PATH)
    old["seed"] = old["seed"].astype(int)
    old["num_cluster"] = old["num_cluster"].astype(int)
    old = old[
        old.apply(lambda row: (int(row["seed"]), int(row["num_cluster"])) in current_targets, axis=1)
    ].copy()
    rows = old.to_dict("records")
    completed = set(zip(old["seed"].astype(int), old["num_cluster"].astype(int)))
else:
    completed = set()

start_all = time.time()
for run_idx, seed in enumerate(SEEDS):
    missing_k = [k for k in K_VALUES if (seed, k) not in completed]
    if not missing_k:
        print(f"Skip completed seed={seed}")
        continue

    print(f"\n=== run {run_idx + 1}/{len(SEEDS)} seed={seed} ===", flush=True)
    set_all_seeds(seed)
    t0 = time.time()

    model, train_loss = train_spadiva(
        X1_input,
        X2_input,
        X1_train,
        X2_train,
        edge_index1=edge_index_fused1,
        edge_weight1=edge_weight_fused1,
        edge_index2=edge_index_fused2,
        edge_weight2=edge_weight_fused2,
        learning_rate=LEARNING_RATE,
        weight=WEIGHT,
        max_epochs=MAX_EPOCHS,
        KL_weight=KL_WEIGHT,
        use_cuda=USE_CUDA,
    )

    Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
        model,
        X1_input,
        X2_input,
        edge_index1=edge_index_fused1,
        edge_weight1=edge_weight_fused1,
        edge_index2=edge_index_fused2,
        edge_weight2=edge_weight_fused2,
        use_cuda=USE_CUDA,
    )

    Z_loc = fuse_shared_latent(Z1_loc, Z2_loc, k=WNN_K)
    Z_W = make_integrated_embedding(Z_loc, W1_loc, W2_loc)
    embeddings = {
        "Z": Z_loc,
        "Z_W": Z_W,
        "W_ADT": W1_loc,
        "W_RNA": W2_loc,
    }

    for k in missing_k:
        row = {
            "run_idx": run_idx,
            "seed": seed,
            "num_cluster": k,
            "max_epochs": MAX_EPOCHS,
            "KL_weight": str(KL_WEIGHT),
            "graph": "spatial+correlation_feature",
            "spatial_k": SPATIAL_K,
            "feature_k": FEATURE_K,
            "spatial_weight": SPATIAL_WEIGHT,
            "feature_weight": FEATURE_WEIGHT,
            "feature_metric": FEATURE_METRIC,
            "final_train_loss": float(train_loss[-1]),
        }
        labels_for_pdf = {}
        for name, emb in embeddings.items():
            ari, nmi, labels = cluster(
                emb,
                num_cluster=k,
                spatial=spatial,
                title=f"{name}_K{k}",
                random_seed=seed,
                show=False,
                Y=Y,
                return_labels=True,
                cluster_method=CLUSTER_METHOD,
            )
            row[f"ARI_{name}"] = float(ari)
            row[f"NMI_{name}"] = float(nmi)
            labels_for_pdf[name] = labels
        rows.append(row)
        completed.add((seed, k))

        if PLOT_EACH_RUN:
            save_spatial_label_pdf(
                spatial,
                labels_for_pdf,
                OUTPUT_DIR / f"spatial_maps_seed{seed}_K{k}.pdf",
                s=30,
            )

        current = pd.DataFrame(rows).drop_duplicates(subset=["seed", "num_cluster"], keep="last")
        current = current.sort_values(["num_cluster", "seed"]).reset_index(drop=True)
        current.to_csv(RAW_PATH, index=False, encoding="utf-8")
        summary = summarize_results(current)
        summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
        print(f"K={k} ARI_Z_W={row['ARI_Z_W']:.6f} NMI_Z_W={row['NMI_Z_W']:.6f}", flush=True)

    print(f"Finished seed={seed} in {time.time() - t0:.1f}s", flush=True)

results_df = pd.DataFrame(rows).drop_duplicates(subset=["seed", "num_cluster"], keep="last")
results_df = results_df.sort_values(["num_cluster", "seed"]).reset_index(drop=True)
results_df.to_csv(RAW_PATH, index=False, encoding="utf-8")
summary_df = summarize_results(results_df)
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")

metadata = {
    "package": "release spaDIVA",
    "max_epochs": MAX_EPOCHS,
    "k_values": K_VALUES,
    "seeds": SEEDS,
    "KL_weight": KL_WEIGHT,
    "graph": "spatial+correlation_feature",
    "spatial_k": SPATIAL_K,
    "feature_k": FEATURE_K,
    "spatial_weight": SPATIAL_WEIGHT,
    "feature_weight": FEATURE_WEIGHT,
    "feature_metric": FEATURE_METRIC,
    "wnn_k": WNN_K,
    "use_cuda": USE_CUDA,
    "torch": torch.__version__,
    "scanpy": sc.__version__,
    "elapsed_seconds": time.time() - start_all,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

summary_df

## 7. Save benchmark boxplots

In [ ]:
save_metric_boxplot(results_df, "ARI", "ARI", ARI_PDF)
save_metric_boxplot(results_df, "NMI", "NMI", NMI_PDF)

print(f"Saved raw results to: {RAW_PATH}")
print(f"Saved summary to: {SUMMARY_PATH}")
print(f"Saved ARI boxplot to: {ARI_PDF}")
print(f"Saved NMI boxplot to: {NMI_PDF}")

## 8. Inspect Z_W results

In [ ]:
results_df[["seed", "num_cluster", "ARI_Z_W", "NMI_Z_W", "final_train_loss"]]

In [ ]:
results_df

In [ ]:
current_targets = {(int(seed), int(k)) for seed in SEEDS for k in K_VALUES}
paper_results_df = results_df.copy()
paper_results_df["seed"] = paper_results_df["seed"].astype(int)
paper_results_df["num_cluster"] = paper_results_df["num_cluster"].astype(int)
paper_results_df = paper_results_df[
    paper_results_df.apply(lambda row: (int(row["seed"]), int(row["num_cluster"])) in current_targets, axis=1)
].copy()

benchmark_columns = [
    "seed", "num_cluster",
    "ARI_Z", "NMI_Z",
    "ARI_Z_W", "NMI_Z_W",
    "ARI_W_ADT", "NMI_W_ADT",
    "ARI_W_RNA", "NMI_W_RNA",
]
paper_results_df[benchmark_columns].to_csv(PAPER_BENCHMARK_CSV, index=False, encoding="utf-8")
print(f"Saved benchmark CSV to: {PAPER_BENCHMARK_CSV}")

if SAVE_PUBLICATION_SPATIAL_MAPS:
    candidates = paper_results_df[paper_results_df["num_cluster"] == PAPER_SPATIAL_MAP_K].copy()
    if candidates.empty:
        raise ValueError(f"PAPER_SPATIAL_MAP_K={PAPER_SPATIAL_MAP_K} is not present in results_df. Current K_VALUES={K_VALUES}.")

    if PAPER_SPATIAL_MAP_SEED in {"best", "best_Z_W"}:
        selected = candidates.sort_values(["ARI_Z_W", "NMI_Z_W", "seed"], ascending=[False, False, True]).iloc[0]
        selected_seed = int(selected["seed"])
    elif PAPER_SPATIAL_MAP_SEED == "median_Z_W":
        median_ari = candidates["ARI_Z_W"].median()
        selected = candidates.assign(_distance=(candidates["ARI_Z_W"] - median_ari).abs()).sort_values(["_distance", "seed"]).iloc[0]
        selected_seed = int(selected["seed"])
    else:
        selected_seed = int(PAPER_SPATIAL_MAP_SEED)
        selected_rows = candidates[candidates["seed"] == selected_seed]
        selected = selected_rows.iloc[0] if not selected_rows.empty else None

    print(f"Saving publication spatial maps with seed={selected_seed}, Z/Z_W K={PAPER_SPATIAL_MAP_K}, W_ADT/W_RNA K={PAPER_SPECIFIC_K}")
    if selected is not None:
        print(f"Selected benchmark row: ARI_Z_W={float(selected['ARI_Z_W']):.6f}, NMI_Z_W={float(selected['NMI_Z_W']):.6f}")

    set_all_seeds(selected_seed)
    model, train_loss = train_spadiva(
        X1_input,
        X2_input,
        X1_train,
        X2_train,
        edge_index1=edge_index_fused1,
        edge_weight1=edge_weight_fused1,
        edge_index2=edge_index_fused2,
        edge_weight2=edge_weight_fused2,
        learning_rate=LEARNING_RATE,
        weight=WEIGHT,
        max_epochs=MAX_EPOCHS,
        KL_weight=KL_WEIGHT,
        use_cuda=USE_CUDA,
    )

    Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
        model,
        X1_input,
        X2_input,
        edge_index1=edge_index_fused1,
        edge_weight1=edge_weight_fused1,
        edge_index2=edge_index_fused2,
        edge_weight2=edge_weight_fused2,
        use_cuda=USE_CUDA,
    )

    Z_loc = fuse_shared_latent(Z1_loc, Z2_loc, k=WNN_K)
    Z_W = make_integrated_embedding(Z_loc, W1_loc, W2_loc)
    z_adata = collect_spadiva_outputs(
        Z=Z_loc,
        W1=W1_loc,
        W2=W2_loc,
        spatial=spatial,
        obs_names=adata_adt.obs_names,
        modality_names=("ADT", "RNA"),
        Z_poe=Z_poe,
        Z1=Z1_loc,
        Z2=Z2_loc,
        X1_hat=X1_hat,
        X2_hat=X2_hat,
    )

    map_specs = [
        ("Z_share", Z_loc, PAPER_SPATIAL_MAP_K, PAPER_OUTPUT_DIR / "Z_share.pdf", "mclust_Z"),
        ("Z_final", Z_W, PAPER_SPATIAL_MAP_K, PAPER_OUTPUT_DIR / "Z_final.pdf", "mclust_Z_W"),
        ("W_ADT", W1_loc, PAPER_SPECIFIC_K, PAPER_OUTPUT_DIR / "W_ADT.pdf", "mclust_W_ADT"),
        ("W_RNA", W2_loc, PAPER_SPECIFIC_K, PAPER_OUTPUT_DIR / "W_RNA.pdf", "mclust_W_RNA"),
    ]

    paper_metrics = {}
    for name, embedding, k, out_pdf, obs_key in map_specs:
        ari, nmi, labels = cluster(
            embedding,
            num_cluster=k,
            spatial=spatial,
            title=f"spaDIVA_{name}",
            random_seed=selected_seed,
            s=30,
            show=False,
            Y=Y,
            return_labels=True,
            cluster_method=CLUSTER_METHOD,
        )
        z_adata.obs[obs_key] = labels
        save_single_spatial_label_pdf(spatial, labels, out_pdf, s=30)
        paper_metrics[name] = (float(ari), float(nmi), int(k), str(out_pdf))

    with open(PAPER_METRICS_TXT, "w", encoding="utf-8") as f:
        f.write(f"selected_seed\t{selected_seed}\n")
        f.write(f"max_epochs\t{MAX_EPOCHS}\n")
        f.write(f"KL_weight\t{KL_WEIGHT}\n")
        f.write(f"graph\tspatial+correlation_feature\n")
        f.write(f"spatial_weight\t{SPATIAL_WEIGHT}\n")
        f.write(f"feature_weight\t{FEATURE_WEIGHT}\n")
        for name, (ari, nmi, k, out_pdf) in paper_metrics.items():
            f.write(f"{name}\tK={k}\tARI={ari:.6f}\tNMI={nmi:.6f}\t{out_pdf}\n")

    print(f"Saved publication spatial maps to: {PAPER_OUTPUT_DIR}")
    print(f"Saved publication metrics to: {PAPER_METRICS_TXT}")
    paper_metrics
